In [ ]:
# ── Imports & Config ──────────────────────────────────────────────────
import socket
import ssl
import threading
import json
import uuid
import logging
import time
from queue import Queue, Empty
from enum import Enum

HOST = '0.0.0.0'
PORT = 9999
WORKER_TIMEOUT = 5
HEARTBEAT_INTERVAL = 3

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
log = logging.getLogger('QueueServer')

log.info('Config loaded. HOST=%s PORT=%d', HOST, PORT)

In [ ]:
# ── Data Structures ───────────────────────────────────────────────────
class JobStatus(Enum):
    QUEUED   = 'queued'
    ASSIGNED = 'assigned'
    DONE     = 'done'


class Job:
    def __init__(self, payload: dict):
        self.job_id       = str(uuid.uuid4())
        self.payload      = payload
        self.status       = JobStatus.QUEUED
        self.worker_id    = None
        self.result       = None
        self.created_at   = time.time()
        self.assigned_at  = None
        self.completed_at = None

    def __repr__(self):
        return f'<Job {self.job_id[:8]} status={self.status.value} task={self.payload.get("task")}>'


class Worker:
    def __init__(self, conn, addr):
        self.worker_id      = str(uuid.uuid4())
        self.conn           = conn
        self.addr           = addr
        self.current_job_id = None
        self.last_seen      = time.time()

    def is_idle(self):
        return self.current_job_id is None

    def __repr__(self):
        return f'<Worker {self.worker_id[:8]} addr={self.addr} idle={self.is_idle()}>'


log.info('Data structures defined.')

In [ ]:
# ── Queue Manager ─────────────────────────────────────────────────────
class QueueManager:
    def __init__(self):
        self._lock    = threading.Lock()
        self._queue   = Queue()
        self._jobs    = {}
        self._workers = {}

    def add_job(self, payload: dict) -> Job:
        job = Job(payload)
        with self._lock:
            self._jobs[job.job_id] = job
        self._queue.put(job.job_id)
        log.info('Job added: %s', job)
        return job

    def assign_next_job(self, worker: Worker):
        try:
            job_id = self._queue.get_nowait()
        except Empty:
            return None
        with self._lock:
            job = self._jobs.get(job_id)
            if job is None:
                return None
            job.status      = JobStatus.ASSIGNED
            job.worker_id   = worker.worker_id
            job.assigned_at = time.time()
            worker.current_job_id = job_id
            worker.last_seen      = time.time()
        log.info('Job assigned: %s -> %s', job, worker)
        return job

    def complete_job(self, job_id: str, worker_id: str, result: str):
        with self._lock:
            job    = self._jobs.get(job_id)
            worker = self._workers.get(worker_id)
            if job is None:
                log.warning('complete_job: unknown job_id %s', job_id)
                return None
            job.status       = JobStatus.DONE
            job.result       = result
            job.completed_at = time.time()
            if worker:
                worker.current_job_id = None
                worker.last_seen      = time.time()
        log.info('Job completed: %s | result=%s', job, result)
        return job

    def requeue_job(self, job_id: str):
        with self._lock:
            job = self._jobs.get(job_id)
            if job and job.status == JobStatus.ASSIGNED:
                job.status      = JobStatus.QUEUED
                job.worker_id   = None
                job.assigned_at = None
        self._queue.put(job_id)
        log.warning('Job re-queued: %s', job_id)

    def register_worker(self, conn, addr) -> Worker:
        worker = Worker(conn, addr)
        with self._lock:
            self._workers[worker.worker_id] = worker
        log.info('Worker registered: %s', worker)
        return worker

    def remove_worker(self, worker_id: str):
        with self._lock:
            worker = self._workers.pop(worker_id, None)
        if worker is None:
            return
        if worker.current_job_id:
            self.requeue_job(worker.current_job_id)
        log.info('Worker removed: %s', worker)

    def update_worker_heartbeat(self, worker_id: str):
        with self._lock:
            worker = self._workers.get(worker_id)
            if worker:
                worker.last_seen = time.time()

    def get_dead_workers(self) -> list:
        now = time.time()
        with self._lock:
            return [
                wid for wid, w in self._workers.items()
                if (now - w.last_seen) > WORKER_TIMEOUT
            ]

    def status_summary(self) -> dict:
        with self._lock:
            return {
                'queued':   sum(1 for j in self._jobs.values() if j.status == JobStatus.QUEUED),
                'assigned': sum(1 for j in self._jobs.values() if j.status == JobStatus.ASSIGNED),
                'done':     sum(1 for j in self._jobs.values() if j.status == JobStatus.DONE),
                'workers':  len(self._workers),
            }


manager = QueueManager()
log.info('QueueManager initialized.')

In [ ]:
# ── Message Handler ───────────────────────────────────────────────────
def send_msg(conn, msg: dict):
    try:
        data = json.dumps(msg) + '\n'
        conn.sendall(data.encode('utf-8'))
    except Exception as e:
        log.error('send_msg failed: %s', e)


def handle_message(msg: dict, conn, worker):
    msg_type = msg.get('type')

    if msg_type == 'submit':
        payload = msg.get('payload')
        if not isinstance(payload, dict) or 'task' not in payload:
            send_msg(conn, {'type': 'error', 'message': 'invalid payload'})
            return worker
        job = manager.add_job(payload)
        send_msg(conn, {'type': 'ack', 'job_id': job.job_id, 'status': 'queued'})
    
    elif msg_type == 'register':
        worker = manager.register_worker(conn, conn.getpeername())
        send_msg(conn, {'type': 'registered', 'worker_id': worker.worker_id})

    elif msg_type == 'request':
        if worker is None:
            send_msg(conn, {'type': 'error', 'message': 'not registered'})
            return worker
        manager.update_worker_heartbeat(worker.worker_id)
        job = manager.assign_next_job(worker)
        if job:
            send_msg(conn, {'type': 'assign', 'job_id': job.job_id, 'payload': job.payload})
        else:
            send_msg(conn, {'type': 'no_job'})

    elif msg_type == 'complete':
        if worker is None:
            send_msg(conn, {'type': 'error', 'message': 'not registered'})
            return worker
        job_id = msg.get('job_id')
        result = msg.get('result', '')
        manager.update_worker_heartbeat(worker.worker_id)
        job = manager.complete_job(job_id, worker.worker_id, result)
        if job:
            send_msg(conn, {'type': 'ack', 'job_id': job_id, 'status': 'done'})
        else:
            send_msg(conn, {'type': 'error', 'message': 'unknown job_id'})
    elif msg_type == 'status':
        job_id = msg.get('job_id')
        if not job_id:
            send_msg(conn, {'type': 'error', 'message': 'missing job_id'})
            return worker
        with manager._lock:
            job = manager._jobs.get(job_id)
        if job is None:
            send_msg(conn, {'type': 'error', 'message': 'unknown job_id'})
            return worker
        send_msg(conn, {
            'type':   'status_response',
            'job_id': job_id,
            'status': job.status.value,
            'result': job.result
        })
    else:
        send_msg(conn, {'type': 'error', 'message': f'unknown message type: {msg_type}'})

    return worker


log.info('Message handler defined.')

In [ ]:
# ── TCP Server ────────────────────────────────────────────────────────
def handle_connection(conn, addr):
    log.info('New connection: %s', addr)
    worker = None
    buffer = ''
    try:
        while True:
            data = conn.recv(4096)
            if not data:
                break
            buffer += data.decode('utf-8')
            while '\n' in buffer:
                line, buffer = buffer.split('\n', 1)
                line = line.strip()
                if not line:
                    continue
                try:
                    msg = json.loads(line)
                    worker = handle_message(msg, conn, worker)
                except json.JSONDecodeError:
                    log.warning('Bad JSON from %s: %s', addr, line)
                    send_msg(conn, {'type': 'error', 'message': 'invalid JSON'})
    except ConnectionResetError:
        log.warning('Connection reset: %s', addr)
    except Exception as e:
        log.error('Connection error %s: %s', addr, e)
    finally:
        conn.close()
        if worker:
            manager.remove_worker(worker.worker_id)
        log.info('Connection closed: %s', addr)


def watchdog():
    log.info('Watchdog started.')
    while True:
        time.sleep(HEARTBEAT_INTERVAL)
        dead = manager.get_dead_workers()
        for wid in dead:
            log.warning('Watchdog: removing dead worker %s', wid[:8])
            manager.remove_worker(wid)


log.info('TCP server functions defined.')

In [ ]:
# ── Start Secure Server ───────────────────────────────────────────────
t_watchdog = threading.Thread(target=watchdog, daemon=True)
t_watchdog.start()

context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
context.load_cert_chain(certfile="server.crt", keyfile="server.key")

server_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server_sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server_sock.bind((HOST, PORT))
server_sock.listen()

log.info('Secure TLS Server listening on %s:%d', HOST, PORT)

try:
    while True:
        raw_conn, addr = server_sock.accept()
        try:
            conn = context.wrap_socket(raw_conn, server_side=True)
        except ssl.SSLError as e:
            log.warning('SSL handshake failed from %s: %s', addr, e)
            raw_conn.close()
            continue
        t = threading.Thread(target=handle_connection, args=(conn, addr), daemon=True)
        t.start()
        log.info('Active connections: %d | Queue status: %s',
                 threading.active_count() - 2,
                 manager.status_summary())
except KeyboardInterrupt:
    log.info('Server shutting down.')
finally:
    server_sock.close()